# Logistic Regression for Type 2 Diabetes Risk

This notebook trains a **binary logistic regression model** using only NumPy/Pandas on:

`x = [EXERANY2, _BMI5CAT, _SMOKER3, MENTHLTH, SSBFRUT3, ALCDAY4]`

Target definition:
- `y = 1`: presence of Type 2 diabetes risk (`DIABETE4 == 1`)
- `y = 0`: absence of Type 2 diabetes risk (`DIABETE4 == 3`)

The train/test split mirrors the Naive Bayes setup (`test_size=0.2`, `random_state=1`).


In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load and prepare data
feature_cols = ['EXERANY2', '_BMI5CAT', '_SMOKER3', 'MENTHLTH', 'SSBFRUT3', 'ALCDAY4']
target_col = 'DIABETE4'

df = pd.read_csv('diabetes_dataset.csv')
df = df[df[target_col].isin([1, 3])].copy()

# Keep only the requested feature vector + target
df = df[feature_cols + [target_col]].dropna().copy()

X = df[feature_cols].astype(float).to_numpy()
y_raw = df[target_col].to_numpy()

# Problem formulation: y in {0,1}
# 1 -> diabetes risk present, 3 -> diabetes risk absent
y = (y_raw == 1).astype(float)

print('Rows used:', len(df))
print('Positive class rate (y=1):', y.mean())

Rows used: 100520
Positive class rate (y=1): 0.14436927974532432


In [3]:
def manual_train_test_split(X, y, test_size=0.2, random_state=1):
    """
    Reproduce sklearn-style shuffled split settings used in Naive_Bayes:
    test_size=0.2, random_state=1.
    """
    n = X.shape[0]
    n_test = int(np.ceil(test_size * n))

    rng = np.random.RandomState(random_state)
    indices = rng.permutation(n)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = manual_train_test_split(
    X, y, test_size=0.2, random_state=1
)

print('Train size:', X_train.shape[0])
print('Test size :', X_test.shape[0])

Train size: 80416
Test size : 20104


In [4]:
# Standardize features using TRAIN statistics only
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
sigma[sigma == 0] = 1.0

X_train_std = (X_train - mu) / sigma
X_test_std = (X_test - mu) / sigma

print('Feature means (train, standardized):', X_train_std.mean(axis=0).round(4))
print('Feature stds  (train, standardized):', X_train_std.std(axis=0).round(4))

Feature means (train, standardized): [ 0. -0. -0.  0.  0. -0.]
Feature stds  (train, standardized): [1. 1. 1. 1. 1. 1.]


In [5]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true, y_prob, eps=1e-12):
    y_prob = np.clip(y_prob, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1.0 - y_true) * np.log(1.0 - y_prob))


def fit_logistic_regression(X, y, lr=0.05, n_iter=5000, verbose_every=500):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    history = []

    for i in range(n_iter):
        z = X @ w + b
        p = sigmoid(z)

        grad_w = (X.T @ (p - y)) / n_samples
        grad_b = np.mean(p - y)

        w -= lr * grad_w
        b -= lr * grad_b

        if i % verbose_every == 0 or i == n_iter - 1:
            loss = binary_cross_entropy(y, p)
            history.append((i, loss))

    return w, b, history


def predict_proba(X, w, b):
    return sigmoid(X @ w + b)


def predict_label(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


def confusion_counts(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}

In [6]:
# Train model
w, b, history = fit_logistic_regression(
    X_train_std,
    y_train,
    lr=0.05,
    n_iter=5000,
    verbose_every=500,
)

print('Final bias:', round(float(b), 6))
print('Loss history (iteration, loss):')
for it, loss in history:
    print(f'  {it:5d}: {loss:.6f}')

# Evaluate
yhat_train = predict_label(X_train_std, w, b, threshold=0.5)
yhat_test = predict_label(X_test_std, w, b, threshold=0.5)

train_acc = accuracy(y_train, yhat_train)
test_acc = accuracy(y_test, yhat_test)

print('\nTrain accuracy:', round(float(train_acc), 4))
print('Test accuracy :', round(float(test_acc), 4))

print('\nTrain confusion:', confusion_counts(y_train.astype(int), yhat_train))
print('Test confusion :', confusion_counts(y_test.astype(int), yhat_test))

Final bias: -2.00064
Loss history (iteration, loss):
      0: 0.693147
    500: 0.377376
   1000: 0.376080
   1500: 0.376035
   2000: 0.376033
   2500: 0.376033
   3000: 0.376033
   3500: 0.376033
   4000: 0.376033
   4500: 0.376033
   4999: 0.376033

Train accuracy: 0.8546
Test accuracy : 0.8573

Train confusion: {'TP': 113, 'TN': 68613, 'FP': 158, 'FN': 11532}
Test confusion : {'TP': 34, 'TN': 17202, 'FP': 35, 'FN': 2833}


In [7]:
# Coefficient table (on standardized features)
coef_table = pd.DataFrame({
    'feature': feature_cols,
    'weight': w,
    'abs_weight': np.abs(w),
}).sort_values('abs_weight', ascending=False)

coef_table

,feature,weight,abs_weight
1,_BMI5CAT,0.526547,0.526547
5,ALCDAY4,0.345093,0.345093
4,SSBFRUT3,0.336802,0.336802
0,EXERANY2,0.261057,0.261057
2,_SMOKER3,-0.101120,0.101120
3,MENTHLTH,0.040571,0.040571


In [ ]:
undersampling_df = pd.read_csv("data_undersampling.csv")
oversampling_df = pd.read_csv("data_oversampling.csv")
smote_df = pd.read_csv("data_smote.csv")

X_train_under = undersampling_df[feature_cols].astype(float).to_numpy()
y_train_under = (undersampling_df[target_col].to_numpy() == 1).astype(float)

X_train_over = oversampling_df[feature_cols].astype(float).to_numpy()
y_train_over = (oversampling_df[target_col].to_numpy() == 1).astype(float)

X_train_smote = smote_df[feature_cols].astype(float).to_numpy()
y_train_smote = (smote_df[target_col].to_numpy() == 1).astype(float)

X_train_under_std = (X_train_under - mu) / sigma
X_train_over_std = (X_train_over - mu) / sigma
X_train_smote_std = (X_train_smote - mu) / sigma

datasets = {
    "undersampling": (X_train_under_std, y_train_under),
    "oversampling": (X_train_over_std, y_train_over),
    "smote": (X_train_smote_std, y_train_smote),
}

def classification_metrics_lr(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return accuracy, precision, recall, f1_score, tp, fp, fn, tn

results = []

for dataset_name, (X_train_dataset, y_train_dataset) in datasets.items():
    w_ds, b_ds, _ = fit_logistic_regression(
        X_train_dataset,
        y_train_dataset,
        lr=0.05,
        n_iter=5000,
        verbose_every=5000
    )
    
    y_test_pred = predict_label(X_test_std, w_ds, b_ds, threshold=0.5)
    accuracy_ds, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics_lr(y_test, y_test_pred)
    
    results.append({
        "dataset": dataset_name,
        "accuracy": accuracy_ds,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    })

results_df = pd.DataFrame(results, columns=["dataset", "accuracy", "precision", "recall", "f1_score", "tp", "fp", "fn", "tn"])
results_df


**Final Model Selection**

Oversampling won by a hair compared to the other datasets in terms of recall and F1 Score. We will proceed with the oversampling dataset.

**Streamlit Prep**

In [ ]:
best_lr_model_final = {
    "model_name": "Best LR Model Final",
    "dataset_method": "original",
    "w": w,
    "b": float(b),
    "mu": mu,
    "sigma": sigma,
}

best_lr_model_final

**Explore Page Setup**

In [ ]:
# Setup datasets for exploration
datasets = {
    "original": (X_train, y_train)
}

try:
    undersampling_df = pd.read_csv("data_undersampling.csv")
    oversampling_df = pd.read_csv("data_oversampling.csv")
    smote_df = pd.read_csv("data_smote.csv")
    
    for name, df_res in [("undersampling", undersampling_df), ("oversampling", oversampling_df), ("smote", smote_df)]:
        # Extract features and target
        df_clean = df_res[feature_cols + [target_col]].dropna().copy()
        
        # Check target format
        unique_targets = df_clean[target_col].unique()
        if 3 in unique_targets and 1 in unique_targets:
            df_clean = df_clean[df_clean[target_col].isin([1, 3])].copy()
            y_raw_res = df_clean[target_col].to_numpy()
            y_res = (y_raw_res == 1).astype(float)
        else:
            y_res = df_clean[target_col].astype(float).to_numpy()
            
        X_res = df_clean[feature_cols].astype(float).to_numpy()
        datasets[name] = (X_res, y_res)
except FileNotFoundError:
    print("Resampled datasets not found. Only original dataset will be available.")


In [ ]:
lr_explore_models = {}
lr_loss_history_data = {}

for dataset_name, (X_ds, y_ds) in datasets.items():
    lr_explore_models[dataset_name] = {}
    lr_loss_history_data[dataset_name] = {}
    
    # Standardize using the dataset's own statistics
    ds_mu = X_ds.mean(axis=0)
    ds_sigma = X_ds.std(axis=0)
    ds_sigma[ds_sigma == 0] = 1.0
    
    X_ds_std = (X_ds - ds_mu) / ds_sigma
    
    # Train
    w_trained, b_trained, history_trained = fit_logistic_regression(
        X_ds_std,
        y_ds,
        lr=0.05,
        n_iter=5000,
        verbose_every=1000
    )
    
    # Predict on the ORIGINAL scaled test set (using ds_mu and ds_sigma since model expects it)
    X_test_ds_std = (X_test - ds_mu) / ds_sigma
    y_pred = predict_label(X_test_ds_std, w_trained, b_trained, threshold=0.5)
    
    acc = accuracy(y_test, y_pred)
    conf = confusion_counts(y_test.astype(int), y_pred)
    
    tp = conf['TP']
    fp = conf['FP']
    fn = conf['FN']
    tn = conf['TN']
    
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
    
    lr_explore_models[dataset_name] = {
        "model_name": f"LR ({dataset_name})",
        "dataset_method": dataset_name,
        "w": w_trained,
        "b": float(b_trained),
        "mu": ds_mu,
        "sigma": ds_sigma,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "accuracy": acc,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }
    
    lr_loss_history_data[dataset_name] = history_trained

print("Available models:", list(lr_explore_models.keys()))


In [ ]:
def predict_with_saved_lr(X_new, saved_model):
    X_new = np.asarray(X_new, dtype=float)
    X_new_scaled = (X_new - saved_model["mu"]) / saved_model["sigma"]
    scores = X_new_scaled @ saved_model["w"] + saved_model["b"]
    y_pred_prob = sigmoid(scores)
    y_pred_bin = (y_pred_prob >= 0.5).astype(int)
    # Convert back to original labels (1=risk, 3=no risk)
    y_pred_original = np.where(y_pred_bin == 1, 1, 3)
    return y_pred_original

def get_lr_explore_model(dataset_name, lr_explore_models):
    return lr_explore_models[dataset_name]


**Loss history data for explore page**

In [ ]:
selected_dataset = "original"

import matplotlib.pyplot as plt

# Extract iterations and losses
history = lr_loss_history_data[selected_dataset]
iters = [h[0] for h in history]
losses = [h[1] for h in history]

plt.plot(iters, losses)
plt.title(f"Binary Cross-Entropy Loss: {selected_dataset}")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()
